<a href="https://colab.research.google.com/github/KeertanaOfficial/AgenticAIProjects/blob/main/load_base_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH03/03_03_begin/notebooks/03_03_load_base_model.ipynb)

# Load the base model and test it on a ticket

1.) Load the base model
2.) Fine-tune
3.) Send it a Ridgeline support ticket
4.) Score its reply against the brand voice rules. This is the prompt-only baseline.

**Runtime > Change runtime type > T4 GPU**.

In [4]:
%%capture
# Install Unsloth. This takes a minute or two.
!pip install unsloth

In [5]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
# Cell here copies the course repo into the session and imports a small helper file called Ridgeline that holds the tickets, the policies, the voice rules, and a checker that scores replies against those rules.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH03")

import ridgeline

tickets = {t["id"]: t for t in ridgeline.load_tickets()}
print(f"Loaded {len(tickets)} Ridgeline tickets.")

Course files already downloaded.
Loaded 18 Ridgeline tickets.


## Load the model

Load Qwen3 4B Instruct in 4-bit precision, the QLoRA setup, then switch Unsloth into inference mode.

In [8]:
#The base model is compressed to 4 bits so it sits comfortably on its GPU.
from unsloth import FastLanguageModel

model,tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,
    load_in_4bit = True
)

model = FastLanguageModel.for_inference(model)



==((====))==  Unsloth 2026.11.9: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Write a helper that generates a reply

`apply_chat_template` wraps the messages in the exact format this model was trained on. Then the model generates new tokens, and you decode only the part it added.

In [9]:
def generate(messages, max_new_tokens=400):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,do_sample=True,temperature=0.7,top_p=0.8)
    #response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


## Build the prompt for one ticket

The baseline prompt carries everything: the instructions, the policies, and the full voice rules.

In [13]:
ticket = tickets["RT-001"]
print(f"Ticket {ticket['customer_name']}: {ticket['subject']}")
print(ticket["message"])

messages = ridgeline.build_messages(ticket, include_voice_rules=True)
prompt_tokens = ridgeline.count_tokens(tokenizer,messages,add_generation_prompt=True)
print(f"\nPrompt tokens: {prompt_tokens} tokens, sent with every ticket")

Ticket Maya Torres: wheres my stuff
ordered a week ago and nothing. RO-48213

Prompt tokens: 773 tokens, sent with every ticket


## Generate a reply

In [14]:
reply = generate(messages)
print(reply)


Both `max_new_tokens` (=400) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Category: Order Status  
Escalate: No  

Hi Maya,  

Your Cascade 65 backpack (RO-48213) shipped from Reno 2 days ago via standard ground. Delivery is projected for within the next 2 business days. The tracking number is 1ZG89234567890.  

You’ll receive a delivery notification in your email. If you don’t get it within 48 hours, check your spam folder and open the carrier trace.  

Happy trails,  
The Ridgeline Crew


## Score the reply

The checker compares the triage headers with the correct answer and tests the reply against every voice rule.

In [16]:
ridgeline.print_check(ridgeline.check_reply(ticket,reply))

RT-001  triage: correct
  voice: passed every rule


## Try two harder tickets

Both of these should be escalated: an injury, and a chargeback threat.

In [18]:
for ticket_id in ["RT-008","RT-017"]:
    ticket = tickets[ticket_id]
    reply = generate(ridgeline.build_messages(ticket, include_voice_rules=True))
    print(f"Ticket {ticket_id} - {ticket['customer_name']}: {ticket['subject']}")
    print(ticket["message"])
    ridgeline.print_check(ridgeline.check_reply(ticket,reply))
    print(reply)
    print()

Both `max_new_tokens` (=400) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ticket RT-008 - Ashley Nguyen: Sole came off mid-hike, I got hurt
The sole of my Talus boot literally peeled off while I was coming down a rocky section. I rolled my ankle badly trying to catch myself. These boots are SIX MONTHS old. This is not okay.
RT-008  triage: correct
  voice: passed every rule
Category: Warranty & Repairs  
Escalate: Yes  

Hi Ashley,  

Your Talus boot failing mid-hike and resulting in an injury is a serious safety concern. This incident has been escalated to our Safety and Quality team for immediate review. We’ll send a specialist to follow up within 24 hours. Please keep the boots and any photos for our inspection. Your safety is our top priority.  

Happy trails,  
The Ridgeline Crew

Ticket RT-017 - Derek Stone: CANCELLED AND STILL CHARGED
I CANCELLED THIS ORDER LAST WEEK AND YOU CHARGED ME $89 ANYWAY. IF THIS IS NOT REFUNDED BY FRIDAY I AM CALLING MY BANK AND DISPUTING THE CHARGE.
RT-017  triage: correct
  voice: passed every rule
Category: Billing & Acco

## What to notice

- The model follows instructions, but you have to spell out every rule on every call.
- Note which rules it missed and whether it escalated the injury and the chargeback threat.
- Read each reply for details that aren't in the ticket or the support system context, like a tracking number. The checker scores format and voice, so it can't catch made-up facts.
- Run the cells again. Some results change from run to run.

Fine-tuning moves the Ridgeline voice into the model itself. That's what the rest of the course builds.